# GraphOPF code

In [1]:
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu121.html

# !pip install pypower
# !pip install pyrlu
# # !pip install conflictfree

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
import torch

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = '0, 1, 2, 3'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)  # 출력결과: cuda
print('Count of using GPUs:', torch.cuda.device_count())   #출력결과: 1 (GPU #2 한개 사용하므로)
print('Current cuda device:', torch.cuda.current_device())  # 출력결과: 2 (GPU #2 의미)

Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [2]:
# %cd /content/drive/MyDrive/kj/GOC4601_case_real_slack
# !pwd

In [2]:
import pickle

from utils.utils4601_graphlde import ACOPFProblem
# from utils.utils4601_graphlde_test import ACOPFProblem

filepath = './data/FeasiblePairs_Case4601_20_perturb_10000_samples.mat'

data = ACOPFProblem(filename=filepath) # call ACOPFProblem class in the utils.py <== In DeepLDE code, need to modify! so messy...

save_data = False

# check the size of train/validation/test dataset.
# print("Dataset of GraphLDE: ")
# print(data.train_dataset)
# print(problem.valid_dataset)
# print(problem.test_dataset)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
data._device = DEVICE
# Put all variables in "data" to the cuda.
for attr in dir(data):
    var = getattr(data, attr)
    if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
        try:
            setattr(data, attr, var.to(DEVICE))
        except AttributeError:
            pass


/global/u1/k/kjsong/FedOPF-APPFL/fine-tuning-task/unseen/Case4601/utils/utils4601_graphlde.py:109: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],


In [3]:
import torch
import torch_geometric
torch.cuda.empty_cache()
import torch.optim as optim
torch.set_default_dtype(torch.float32) #  If the inputs are torch.float32, must be torch.complex64. If the inputs are torch.float64, must be torch.complex128.

from torch.utils.data import TensorDataset, DataLoader, Dataset

import numpy as np
import pickle
import time
import os
import random

from pypower.api import loadcase

from model.Edge_GNN_solver import Edge_GNNSolver

from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg
from global_config import base_config, global_logger, ROOT_DIRECTORY, logging
from pathlib import Path
import pickle

# from conflictfree.grad_operator import ConFIG_update
# from conflictfree.momentum_operator import PseudoMomentumOperator
# # from conflictfree.grad_operator import ConFIGOperator
# from conflictfree.utils import get_gradient_vector,apply_gradient_vector

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Set the linear algebra solver(?)
torch._C._set_linalg_preferred_backend(torch._C._LinalgBackend.Magma) # torch._C._LinalgBackend.Magma, Cusolver
torch._C._get_linalg_preferred_backend()

# MAGMA_NOWARNING
# torch.set_warn_always(False)

[W1222 21:15:44.752995270 Context.cpp:320] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


<_LinalgBackend.Magma: 2>

* First, let's see the information of ACOPF for targeted power network!

In [4]:
ppc = loadcase("./data/matpower/pglib_opf_case4601_goc.mat") # in this code, we used Pypower for loading benchmark power network.
## NOTE: the dataset we used are Pypower and PGLib, so we additionally need to check whether the targeted power network is same or not!
## e.g., IEEE 57case in Pypower has different "rate A", "rate B", "rate C" values compared to PGLib.

ng = ppc['gen'].shape[0] # number of generators.
nbus = ppc['bus'].shape[0] # number of buses.
nl = ppc['branch'].shape[0] # total number of branches and transformers.

In [14]:
from scipy.stats.qmc import LatinHypercube
train_config = {
    'probType': 'acopf',
    'useCompl': True, # boolean type: whether to use completion (DC3, DeepLDE 기술)

    # GNN model parameters
    'n_gnn_layers': 3,
    'nfeature_dim': 2, # input dim
    'efeature_dim': 4, # edge feature dim 
    'hidden_dim': 40,
    'dropout_rate': 0.1,
    'K': 10, # 6 # only for TAGConv or ChebConv and GATConv (as multi-head)

    # DeepLDE hyperparameters
    'epochs': 30, # 10 (GPU RTX 4090), 8 (GPU A100)
    'batchSize': 5, # 6, (8) (GPU RTX 4090; GPU 다운 에러 발생..), 16 (GPU A100)
    'lr': 1e-3, # 1e-3
    'lr_w': 1e-3, # 1e-3
    'weight_decay': 1e-5,

    # LDF parameters
    'rho_init': 0.001, # 0.001, 
    's_init': 0.1,
    'p_iter_max': 10,
    'warmup_iter': 0, # 20, # 0 for non-warmup start cases
    'corrEps': 1e-4, # float type: correction procedure tolerance
}

eps_converge = train_config['corrEps']
valid_eps_converge = 1e-4
nepochs = train_config['epochs']
batch_size = train_config['batchSize']

train_loss_list = []
valid_loss_list = []
valid_eval_list = []

Kshot = 20
train_len_range = (0,Kshot) ## K-shot: 20, 10, 5, 1, 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(train_len_range) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

## Random sampling
# train_loader = torch_geometric.loader.DataLoader(random.sample(data.train_dataset, 100), batch_size=train_config['batchSize'], shuffle=True, drop_last=True)
## Slicing
train_loader = torch_geometric.loader.DataLoader(data.train_dataset[train_len_range[0]:train_len_range[1]], batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

# ## Latin hypercube sampling
# data__ = list(range(1,101))
# n_samples = 100
# sampler = LatinHypercube(d=1)
# samples = sampler.random(n=n_samples)
# indices = np.floor(samples*len(data__)).astype(int).flatten()
# trained_sample_data = [data.train_dataset[i] for i in indices]
# train_loader = torch_geometric.loader.DataLoader(trained_sample_data, batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

valid_loader = torch_geometric.loader.DataLoader(data.valid_dataset, batch_size=train_config['batchSize'], shuffle=False, drop_last=True)

# solver_net = GNNSolver(data, train_config)
solver_net = Edge_GNNSolver(data, train_config)

solver_net.to(DEVICE)

print(solver_net)
num_params = sum(p.numel() for p in solver_net.parameters() if p.requires_grad)
print('The number of parameters of model is', num_params)

Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=15960, out_features=265, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
The number of parameters of model is 4691985


In [15]:
#################################### SETTING THE LOGS ####################################
result_path = os.path.join(ROOT_DIRECTORY, "results")
# folder_name = "FT_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
#               + str(train_config["rho_init"]) + "_rho_init"
folder_name = "Local_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
               + str(train_config["rho_init"]) + "_rho_init"

log_path = os.path.join(result_path, folder_name, "logs")
data_tracking_path = os.path.join(result_path, folder_name, "data_tracking")

Path(log_path).mkdir(parents=True, exist_ok=True)
Path(data_tracking_path).mkdir(parents=True, exist_ok=True)
fileh = logging.FileHandler(os.path.join(log_path, "log.txt"), 'a')
global_logger.addHandler(fileh)

* Load pretrained GraphOPF model

In [7]:
pretrained_global_graphlde = torch.load("./model/global_model/checkpoint_Global.pth", weights_only=False)

* Transfer the weights from the pretrained GraphOPF

In [8]:
global_pretrained_state_dict = pretrained_global_graphlde
target_state_dict = solver_net.state_dict()

layer_names = list(target_state_dict.keys())
# load global model
for name in layer_names[:-2]:
    if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
        # print(name)
        target_state_dict[name] = global_pretrained_state_dict[name].clone()

solver_net.load_state_dict(target_state_dict)

# # Freezing the half-GNN layers
# for i, (name, param) in enumerate(solver_net.named_parameters()):
#     if i <= 32: # 5, 10, 21
#         # print(name)
#         param.requires_grad = False
#     else:
#         print(name)
#         param.requires_grad = True

<All keys matched successfully>

* Training method: LD framework

In [16]:
stats = {}

# NOTE: LDF parameters.
LagM_sp_gen = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_gen = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_bus = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

warmup_iter = train_config["warmup_iter"] # the warmup period: the NN is trained with an additional inner iteration before the first outer iteration.
rho_init = train_config["rho_init"]
s_init = train_config["s_init"]

rho = rho_init
s = s_init

rho_iter = 0
s_iter = 0

p_iter_max = train_config["p_iter_max"]
p_iter_max_sum = p_iter_max

d = 0 # 0 for static case otherwise use 5
beta = 0 # 0.001 

lr_w = train_config["lr_w"] # 이거 증가해도 되지 않을지?
lr = train_config["lr"]

print_interval = 1
epoch_stats = {}
for i in range(nepochs):

    ################### TRAINING PHASE ###################
    solver_net.train()
    if i<warmup_iter:
        ######### WARM-UP PERIOD #########
        if i == 0:
            solver_opt = optim.Adam(solver_net.parameters(), lr=lr_w, weight_decay=train_config["weight_decay"]) # this will be reinitalized after warmup stage
            print("Warmup start!")

        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            # start_time = time.time()
            solver_opt.zero_grad()

            # DNN + NR (equality constraint)
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    else:
        if i == warmup_iter:
            print("Warmup ended!")
            # data.ref_freedom = False

            # ineq_lag_flag = True # for the first warmup end epoch, consider lag. multipliers update of ineq.
        # elif (i - warmup_iter)%ineq_lag_flag_trigger == 0:
        #     # print("Doing test... continue this case..")
        #     print("consider lagrangian multipliers update for ineq. constraints!")
        #     ineq_lag_flag = True
        # else:
        #     ineq_lag_flag = False

        # for every (updated) p_iter_max_sum time.
        if (i - warmup_iter)%p_iter_max_sum == 0:
            ######### Outer Interation: calculate step size of lagrangian multipliers update #########
            if i> warmup_iter:
                print("current epoch %d || p_iter_max updated : %d -> %d" %(i, p_iter_max, p_iter_max + d))
                # s = s_init * (1/(1+beta*(s_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                # s_iter += 1
                # print("mu iter updated : %d -> %d" %(s_iter-1, s_iter))

                rho = rho_init * (1/(1+beta*(rho_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                rho_iter += 1
                print("rho iter updated : %d -> %d" %(rho_iter-1, rho_iter))
                p_iter_max = p_iter_max + d
                p_iter_max_sum += p_iter_max

            with torch.no_grad():
                print("Lambda updated at %d epoch" %i)
                solver_net.eval()
                for Xtrain in train_loader:
                    Xtrain = Xtrain.to(DEVICE)
                    #solver_opt.zero_grad()
                    Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

                    LagM_sp_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[:2] # Update the lagrangian multiplier.
                    LagM_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2:2+2*ng] # Update the lagrangian multiplier.
                    LagM_bus += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng:2+2*ng+2*nbus] # Update the lagrangian multiplier.
                    LagM_line += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng+2*nbus:] # Update the lagrangian multiplier.

            solver_opt = optim.Adam(solver_net.parameters(), lr = lr, weight_decay=train_config["weight_decay"])

        solver_net.train()
        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            solver_opt.zero_grad()
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, obj_train, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    if (i == 0) or (i%print_interval == 0):
        print(
            'Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    global_logger.info('Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    train_loss_list.append(np.mean(epoch_stats['train_loss']))
    # valid_eval_list.append(np.mean(epoch_stats['valid_eval']))

Warmup ended!
Lambda updated at 0 epoch
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are desi

Epoch 0: train loss 291395.4688, train obj 2914.6011, ineq max 1408.0027, ineq mean 0.6389, ineq p_g num viol 1.0000, ineq q_g num viol 54.6000, ineq v_m num viol 7.4500, ineq line_theraml num viol 490.7000, eq max 0.0004, eq mean 0.0000


Epoch 0: train loss 291395.4688, train obj 2914.6011, ineq max 1408.0027, ineq mean 0.6389, ineq p_g num viol 1.0000, ineq q_g num viol 54.6000, ineq v_m num viol 7.4500, ineq line_theraml num viol 490.7000, eq max 0.0004, eq mean 0.0000
 you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical rou

Epoch 1: train loss 178941.8281, train obj 2024.9055, ineq max 979.3488, ineq mean 0.4168, ineq p_g num viol 1.0000, ineq q_g num viol 50.2750, ineq v_m num viol 3.7250, ineq line_theraml num viol 323.2750, eq max 0.0004, eq mean 0.0000


Epoch 1: train loss 178941.8281, train obj 2024.9055, ineq max 979.3488, ineq mean 0.4168, ineq p_g num viol 1.0000, ineq q_g num viol 50.2750, ineq v_m num viol 3.7250, ineq line_theraml num viol 323.2750, eq max 0.0004, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good perfor

...

Epoch 27: train loss 15116.9375, train obj 650.0388, ineq max 281.2411, ineq mean 0.0758, ineq p_g num viol 0.6589, ineq q_g num viol 54.0661, ineq v_m num viol 1.6214, ineq line_theraml num viol 87.2321, eq max 0.0005, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performa

Epoch 28: train loss 14611.2861, train obj 642.7056, ineq max 271.6611, ineq mean 0.0732, ineq p_g num viol 0.6362, ineq q_g num viol 52.5603, ineq v_m num viol 1.5655, ineq line_theraml num viol 84.3190, eq max 0.0005, eq mean 0.0000


Epoch 28: train loss 14611.2861, train obj 642.7056, ineq max 271.6611, ineq mean 0.0732, ineq p_g num viol 0.6362, ineq q_g num viol 52.5603, ineq v_m num viol 1.5655, ineq line_theraml num viol 84.3190, eq max 0.0005, eq mean 0.0000
 might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small size

Epoch 29: train loss 14138.9219, train obj 635.7376, ineq max 262.6758, ineq mean 0.0707, ineq p_g num viol 0.6150, ineq q_g num viol 51.1150, ineq v_m num viol 1.5133, ineq line_theraml num viol 81.5333, eq max 0.0005, eq mean 0.0000


Epoch 29: train loss 14138.9219, train obj 635.7376, ineq max 262.6758, ineq mean 0.0707, ineq p_g num viol 0.6150, ineq q_g num viol 51.1150, ineq v_m num viol 1.5133, ineq line_theraml num viol 81.5333, eq max 0.0005, eq mean 0.0000


In [17]:
# Save the training history
# train_loss_list
# epoch_stats['train_loss']
(np.array(train_loss_list)).tolist()
global_logger.info("train_loss_list:{}".format((np.array(train_loss_list)).tolist()))

data_tracking = {
                "train_loss_list": (np.array(train_loss_list)).tolist(),
                }
with open(os.path.join(data_tracking_path, "metrics.pickle"), 'wb') as handle:
    pickle.dump(data_tracking, handle, protocol=pickle.HIGHEST_PROTOCOL)

train_loss_list:[291395.46875, 178941.828125, 120379.9453125, 91141.625, 74132.5234375, 62733.30078125, 54417.7578125, 48067.3515625, 43034.20703125, 38942.9453125, 36271.7578125, 33890.6328125, 31546.80859375, 29425.81640625, 27555.404296875, 25911.833984375, 24424.07421875, 23112.80078125, 21927.40234375, 20858.404296875, 19938.35546875, 19088.27734375, 18290.57421875, 17550.779296875, 16872.146484375, 16242.48828125, 15658.96875, 15116.9375, 14611.2861328125, 14138.921875]


* Evaluation (using validation set)

In [18]:
len(data.test_dataset)

200

In [19]:
# load pretrained_model
# solver_net = torch.load(r'./models/pretrained_models/graphlde_4601_pretrained_model_20_real_slack_chebconv.pt', weights_only=False)
# solver_net = torch.load(r'./models/pretrained_models/graphlde_4601_pretrained_model_20_real_slack_chebconv_(weight_init_default).pt', weights_only=False)


In [19]:
from pypower.api import makeYbus
Ybus, Yf, Yt = makeYbus(data.baseMVA, data.ppc['bus'], data.ppc['branch'])
# branch thermal limit information
flow_max = (data.ppc['branch'][:, 5] / data.baseMVA)**2
flow_max[flow_max == 0] = np.inf
flow_max = torch.tensor(flow_max, dtype=torch.float32).to(data.device)

test_len = 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

test_loader = torch_geometric.loader.DataLoader(data.test_dataset[test_len:], batch_size=1, shuffle=False, drop_last=True)
# test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

solver_net.eval()
test_stats = {}
test_eps_converge = 1e-4

LagM_sp_g = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_q_g = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_v_m = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line_l = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

solve_time = []
for (i, Xtest) in enumerate(test_loader):
    Xtest = Xtest.to(DEVICE)

    start_time = time.time()
    Y = solver_net(Xtest, n_means, n_stds, e_means, e_stds)
    # Y = solver_net(Xtest, e_means, e_stds)
    end_time = time.time()

    solve_time += [end_time - start_time]

    ## line thermal limit
    pg, qg, vm, va = data.get_yvars(Y)
    vr = vm*torch.cos(va)
    vi = vm*torch.sin(va)
    vz = torch.complex(vr, vi) # complex voltage

    # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
    If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(data.device) @ vz.T
    It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(data.device) @ vz.T

    # Calculate the apparent power S
    Sf = vz[:,data.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
    St = vz[:,data.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
    Sff = Sf * torch.conj(Sf)
    Stt = St * torch.conj(St)

    # calculate the line thermal limit constraints violation
    diff_Sf = Sff.real - flow_max
    diff_St = Stt.real - flow_max
    # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

    line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
    line_limit_vio_St = torch.clamp(diff_St, 0)
    ###########################################

    test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

    dict_agg(test_stats, 'time', end_time - start_time, op='sum')

    test_ineq_p_g = torch.cat([pg - data.pmax, data.pmin - pg], dim=1)
    test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(data.device)
    test_ineq_q_g = test_ineq_dist[:,2:2+2*ng]
    test_ineq_v_m = test_ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
    test_ineq_line_l = test_ineq_dist[:,2+2*ng+2*nbus:]

    dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
    # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

    dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

    pg_rate_torch = (((pg <= data.pmax) & (pg >= data.pmin)).sum()/ng)*100
    qg_rate_torch = (((qg <= data.qmax) & (qg >= data.qmin)).sum()/ng)*100
    dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

    v_rate_torch = (((vm <= data.vmax) & (vm >= data.vmin)).sum()/nbus)*100
    dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

    sff_rate_torch = ((Sff.real <= flow_max).sum()/nl)*100        
    stt_rate_torch = ((Stt.real <= flow_max).sum()/nl)*100        
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

    test_eq_real = test_eq_resid[:,:nbus]
    test_eq_react = test_eq_resid[:,nbus:]
    dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:nbus] <= 1e-2) & (test_eq_resid[:,:nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:nbus].shape[1]*100).detach().cpu().numpy())
    dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,nbus:] <= 1e-2) & (test_eq_resid[:,nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,nbus:].shape[1]*100).detach().cpu().numpy())

    print('Test batch {}: test loss {:.4f}, test obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}, p_g satisfication rate {:.4f}, q_g satisfication rate {:.4f}, v_m satisfication rate {:.4f}, test_line_limit_satisfication_rate {:.4f}, test_line_limit_satisfication_rate_Sf {:.4f}, test_line_limit_satisfication_rate_St {:.4f}, active eq satisfication rate {:.4f}, reactive eq satisfication rate {:.4f}'.format(
                i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_obj_cost']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
                np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']),
                np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean']), np.mean(test_stats['test_p_g_satisfication rate (%)']), np.mean(test_stats['test_q_g_satisfication rate (%)']), np.mean(test_stats['test_v_m_satisfication rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']), np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_active_eq_satisfication rate (%)']), np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))


Test batch 0: test loss 442.6135, test obj 433.8425, ineq max 2.1374, ineq mean 0.0004, ineq q_g num viol 10.0000, ineq v_m num viol 0.0000, eq max 0.0003, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 92.4812, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.9861, test_line_limit_satisfication_rate_Sf 99.9861, test_line_limit_satisfication_rate_St 99.9861, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 1: test loss 442.8206, test obj 433.9777, ineq max 2.1388, ineq mean 0.0004, ineq q_g num viol 9.5000, ineq v_m num viol 0.0000, eq max 0.0006, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 92.8571, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.9792, test_line_limit_satisfication_rate_Sf 99.9792, test_line_limit_satisfication_rate_St 99.9792, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 2: test loss 440

* Arithmetic mean

In [20]:
## Calculate the results of GraphLDE

print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

print("\n")
print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

print("\n")
print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
# print("DeepLDE time (ms) <== average value for test dataset:", (test_stats['time']/1000)*1e3)
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)

global_logger.info('GraphLDE obj. value for test samples: {}'.format(round(np.mean(test_stats['test_obj_cost'])*10000, 4)))
global_logger.info('GraphLDE eq. mean for test samples: {}'.format(np.mean(test_stats['test_eq_mean'])))
global_logger.info('GraphLDE eq. max for test samples: {}'.format(np.mean(test_stats['test_eq_max'])))
global_logger.info('GraphLDE eq. active mean for test samples: {}'.format(np.mean(test_stats['test_eq_real_mean'])))
global_logger.info('GraphLDE eq. active max for test samples: {}'.format(np.mean(test_stats['test_eq_real_max'])))
global_logger.info('GraphLDE eq. reactive mean for test samples: {}'.format(np.mean(test_stats['test_eq_react_mean'])))
global_logger.info('GraphLDE eq. reactive max for test samples: {}'.format(np.mean(test_stats['test_eq_react_max'])))
global_logger.info('\nGraphLDE ineq. mean for test samples: {}'.format(np.mean(test_stats['test_ineq_mean'])))
global_logger.info('GraphLDE ineq. max for test samples: {}'.format(np.mean(test_stats['test_ineq_max'])))
global_logger.info('GraphLDE ineq. p_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_mean'])))
global_logger.info('GraphLDE ineq. p_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_max'])))
global_logger.info('GraphLDE ineq. q_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_mean'])))
global_logger.info('GraphLDE ineq. q_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_max'])))
global_logger.info('GraphLDE ineq. v_m mean for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_mean'])))
global_logger.info('GraphLDE ineq. v_m max for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_max'])))
global_logger.info('GraphLDE ineq. line_l mean for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_mean'])))
global_logger.info('GraphLDE ineq. line_l max for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_max'])))
global_logger.info('\nGraphLDE p_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_p_g_satisfication rate (%)'])))
global_logger.info('GraphLDE q_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_q_g_satisfication rate (%)'])))
global_logger.info('GraphLDE v_m satisfication rate for test samples: {}'.format(np.mean(test_stats['test_v_m_satisfication rate (%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_Sf for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_St for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_St(%)'])))
global_logger.info('GraphLDE active eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_active_eq_satisfication rate (%)'])))
global_logger.info('GraphLDE reactive eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))
global_logger.info('\nGraphLDE time (ms) <== average value for test dataset: {}'.format((np.mean(solve_time)/1)*1e3))


GraphLDE obj. value for test samples: 4310591.5
GraphLDE eq. mean for test samples: 5.69680696571595e-06
GraphLDE eq. max for test samples: 0.00046060956083238125
GraphLDE eq. active mean for test samples: 2.775501343421638e-06
GraphLDE eq. active max for test samples: 0.00012380423140712082
GraphLDE eq. reactive mean for test samples: 8.618112588010263e-06
GraphLDE eq. reactive max for test samples: 0.00046060956083238125

GraphLDE ineq. mean for test samples: 0.0003703230759128928
GraphLDE ineq. max for test samples: 2.138141393661499
GraphLDE ineq. p_g mean for test samples: 0.0
GraphLDE ineq. p_g max for test samples: 0.0
GraphLDE ineq. q_g mean for test samples: 0.028071625158190727
GraphLDE ineq. q_g max for test samples: 2.138141393661499
GraphLDE ineq. v_m mean for test samples: 0.0
GraphLDE ineq. v_m max for test samples: 0.0
GraphLDE ineq. line_l mean for test samples: 9.527839574730024e-05
GraphLDE ineq. line_l max for test samples: 0.6854665875434875

GraphLDE p_g satisfica

GraphLDE obj. value for test samples:  4310591.5
GraphLDE eq. mean for test samples:  5.696807e-06
GraphLDE eq. max for test samples:  0.00046060956
GraphLDE eq. active mean for test samples:  2.7755013e-06
GraphLDE eq. active max for test samples:  0.00012380423
GraphLDE eq. reactive mean for test samples:  8.618113e-06
GraphLDE eq. reactive max for test samples:  0.00046060956


GraphLDE ineq. mean for test samples:  0.00037032308
GraphLDE ineq. max for test samples:  2.1381414
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.028071625
GraphLDE ineq. q_g max for test samples:  2.1381414
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  9.5278396e-05
GraphLDE ineq. line_l max for test samples:  0.6854666


GraphLDE p_g satisfication rate for test samples:  100.0
GraphLDE q_g satisfication rate for test samples

* Harmonic mean

In [9]:
import statistics as st

## Calculate optimality gap
print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", st.harmonic_mean(test_stats['test_eq_mean'])) # print("LDF eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", st.harmonic_mean(test_stats['test_eq_max'])) # print("LDF eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", st.harmonic_mean(test_stats['test_eq_real_mean'])) # print("LDF eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", st.harmonic_mean(test_stats['test_eq_real_max'])) # print("LDF eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", st.harmonic_mean(test_stats['test_eq_react_mean'])) # print("LDF eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", st.harmonic_mean(test_stats['test_eq_react_max'])) # print("LDF eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))
print("\n")
print("GraphLDE ineq. mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_mean'])) # print("LDF ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", st.harmonic_mean(test_stats['test_ineq_max'])) # print("LDF ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_mean'])) # print("LDF ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_max'])) # print("LDF ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_mean'])) # print("LDF ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_max'])) # print("LDF ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_mean'])) # print("LDF ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_max'])) # print("LDF ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_mean'])) # print("LDF ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_max'])) # print("LDF ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))
print("\n")

print("GraphLDE p_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_p_g_satisfication rate (%)'].reshape(-1))) # print("LDF p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_q_g_satisfication rate (%)'].reshape(-1))) # print("LDF q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_v_m_satisfication rate (%)'].reshape(-1))) # print("LDF v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_St(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_active_eq_satisfication rate (%)'].reshape(-1))) # print("LDF active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_reactive_eq_satisfication rate (%)'].reshape(-1))) # print("LDF reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)

global_logger.info('GraphLDE obj. value for test samples: {}'.format(round(np.mean(test_stats['test_obj_cost'])*10000, 4)))
global_logger.info('GraphLDE eq. mean for test samples: {}'.format(np.mean(test_stats['test_eq_mean'])))
global_logger.info('GraphLDE eq. max for test samples: {}'.format(np.mean(test_stats['test_eq_max'])))
global_logger.info('GraphLDE eq. active mean for test samples: {}'.format(np.mean(test_stats['test_eq_real_mean'])))
global_logger.info('GraphLDE eq. active max for test samples: {}'.format(np.mean(test_stats['test_eq_real_max'])))
global_logger.info('GraphLDE eq. reactive mean for test samples: {}'.format(np.mean(test_stats['test_eq_react_mean'])))
global_logger.info('GraphLDE eq. reactive max for test samples: {}'.format(np.mean(test_stats['test_eq_react_max'])))
global_logger.info('\nGraphLDE ineq. mean for test samples: {}'.format(np.mean(test_stats['test_ineq_mean'])))
global_logger.info('GraphLDE ineq. max for test samples: {}'.format(np.mean(test_stats['test_ineq_max'])))
global_logger.info('GraphLDE ineq. p_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_mean'])))
global_logger.info('GraphLDE ineq. p_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_max'])))
global_logger.info('GraphLDE ineq. q_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_mean'])))
global_logger.info('GraphLDE ineq. q_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_max'])))
global_logger.info('GraphLDE ineq. v_m mean for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_mean'])))
global_logger.info('GraphLDE ineq. v_m max for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_max'])))
global_logger.info('GraphLDE ineq. line_l mean for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_mean'])))
global_logger.info('GraphLDE ineq. line_l max for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_max'])))
global_logger.info('\nGraphLDE p_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_p_g_satisfication rate (%)'])))
global_logger.info('GraphLDE q_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_q_g_satisfication rate (%)'])))
global_logger.info('GraphLDE v_m satisfication rate for test samples: {}'.format(np.mean(test_stats['test_v_m_satisfication rate (%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_Sf for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_St for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_St(%)'])))
global_logger.info('GraphLDE active eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_active_eq_satisfication rate (%)'])))
global_logger.info('GraphLDE reactive eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))
global_logger.info('\nGraphLDE time (ms) <== average value for test dataset: {}'.format((np.mean(solve_time)/1)*1e3))

GraphLDE obj. value for test samples:  3459218.4448
GraphLDE eq. mean for test samples:  6.228278545185801e-06
GraphLDE eq. max for test samples:  0.0003987515364172608
GraphLDE eq. active mean for test samples:  2.9959126176184707e-06
GraphLDE eq. active max for test samples:  0.00010875316994758176
GraphLDE eq. reactive mean for test samples:  9.458952113910127e-06
GraphLDE eq. reactive max for test samples:  0.0003986132026868001


GraphLDE ineq. mean for test samples:  6.148506295824064e-07
GraphLDE ineq. max for test samples:  0.010572804678306503
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  5.3364120426584116e-05
GraphLDE ineq. q_g max for test samples:  0.010449723644732631
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisficatio

/home/super/anaconda3/envs/skjdeep/lib/python3.10/statistics.py:428: RuntimeWarning: divide by zero encountered in divide
  T, total, count = _sum(w / x if w else 0 for w, x in zip(weights, data))


In [10]:
############ UNIFORM DIST. WITH RANDOM SAMPLING ############
# MATPOWER - cost: 3435500, solve time: 5489.90 ms  <=== +/- 20% load perturbation
# GraphLDE - cost: 3459218.4448, solve time: 179.27332890033722 ms <=== +/- 20% load perturbation <=== training time: ~ 38min (37min 51s)

((3459218.4448 - 3435500)/3435500)*100 # <== 20% load perturbation

0.6903928045408287